# 🕵️‍♂️ Yüz Algılama Uygulaması

Bu proje, bilgisayardan seçilen bir resimdeki insan yüzlerini **OpenCV** kütüphanesi kullanarak otomatik olarak bulan ve işaretleyen bir masaüstü uygulamasıdır.

Aşağıdaki hücreleri **sırasıyla (Shift + Enter)** çalıştırarak kodun nasıl adım adım inşa edildiğini inceleyebilirsiniz.

## 1. Adım: Gerekli Kütüphanelerin İçe Aktarılması ve Modelin Yüklenmesi

Bu projede kullanacağımız kütüphanelerin her birinin özel bir görevi vardır. Eğer bilgisayarınızda yüklü değillerse terminale (komut satırına) ilgili kurulum kodlarını yazarak indirebilirsiniz:

* **`cv2` (OpenCV)**: Resimleri okumak, üzerinde çizim yapmak (dikdörtgen çizmek) ve içindeki yapay zeka algoritmalarıyla resimdeki **yüzleri tespit etmek** için kullanılır. Görüntü işlemenin kalbidir.
  * *Nasıl İndirilir:* `pip install opencv-python`
* **`tkinter`**: Python'un standart görsel arayüz (GUI) oluşturucusudur. Pencereler, butonlar ve kullanıcının resim seçeceği "Dosya Seç" diyaloglarını (`filedialog`) oluşturmak için kullanırız.
  * *Nasıl İndirilir:* Python ile birlikte kurulu gelir, ekstra indirmeye gerek yoktur.
* **`PIL` (Pillow)**: OpenCV'nin işlediği resim formatı, Tkinter arayüzünde doğrudan gösterilemez. Bu yüzden resmi arayüzün anlayacağı formata çevirmek ve arayüze sığacak şekilde kaliteli bir biçimde küçültmek (resize) için `PIL` kütüphanesini kullanırız.
  * *Nasıl İndirilir:* `pip install Pillow`
* **`numpy`**: Normal şartlarda `cv2.imread()` kodu resmi bilgisayardan rahatça okuyabilir ancak Windows sistemlerinde seçilen resmin klasör yolunda Türkçe karakter (Örn: `Masaüstü/Fotoğraflar/yüz.jpg`) varsa hata verir. Bu engeli aşmak için resmi doğrudan `numpy` kütüphanesi ile ham "bayt" (sayısal veri) olarak okuruz.
  * *Nasıl İndirilir:* `pip install numpy`

Ayrıca aşağıdaki kodda `cv2` içerisinde gelen önceden eğitilmiş **Haar Cascade** yüz tanıma modelini (`face_detector.xml`) projenin klasöründen çekip hafızaya alıyoruz.

In [ ]:
import cv2
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import numpy as np

# Önceden eğitilmiş Haar Cascade yüz algılama modelini (XML dosyası) yülüyoruz
face_cascade = cv2.CascadeClassifier('face_detector.xml')

## 2. Adım: Yüz Algılama Fonksiyonunun İnşa Edilmesi

Uygulamamızın beyni olan `open_file` fonksiyonu, arayüzdeki "Dosya Seç" butonuna tıkladığımızda devreye girer. Bu devasa işlemi tek bir satıra dökmek yerine, önce parça parça ne yaptığını ve nasıl kodlandığını aşağıdaki gerçek kod hücrelerinde görelim:

### 2.1. Dosya Seçimi ve Türkçe Karakter Sorununun Çözümü
Öncelikle `filedialog.askopenfilename()` koduyla kullanıcıya bir resim seçtirilir. Seçilen resim, yukarıda bahsettiğimiz Windows'taki Türkçe karakter problemine takılmamak için önce `numpy` ile bayt dizisi olarak okunur. Ardından `cv2.imdecode` ile OpenCV'nin işleyebileceği gerçek bir görüntüye (çözülmüş matrise) dönüştürülür.

In [ ]:
# 2.1 - Dosya Seçimi ve İçe Aktarma
file_path = filedialog.askopenfilename()

# Eğer kullanıcı iptal etmeyip bir dosya seçtiyse:
if file_path:
    # Türkçe karakterleri hatasız okumak için numpy yöntemi
    img_array = np.fromfile(file_path, np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

### 2.2. Görüntüyü Yapay Zekaya Hazırlama (Gri Tonlama ve Histogram)
Haar Cascade yapay zeka modelleri resimdeki **renklerle ilgilenmez**, sadece aydınlık ve karanlık piksellerin oluşturduğu hatlara (gölgelere ve sınırlara) odaklanır. Örneğin göz çukurları her zaman daha karanlık, burun kemiği ise aydınlıktır. 
Bu yüzden bilgisayarın hızlı çalışması için resmi önce **siyah-beyaz tona** çeviririz.
Ayrıca, fotoğrafın bazı yerleri çok aydınlık veya karanlık olup yapay zekayı şaşırtmasın diye ışığı dengeleriz (Histogram Eşitleme).

In [ ]:
# 2.2 - Yapay Zekaya Hazırlık (Griye Çevirme ve Işık Dengeleme)
# (Not: Orijinal fonksiyonda bu işlemler 'if file_path:' şartının içinde gerçekleşir)

# Gri tona çevirme
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# Işık ve gölgeleri tüm resme eşit dağıtma
gray = cv2.equalizeHist(gray)

### 2.3. Yüzleri Tespit Etme ve İşaretleme
Hazırladığımız bu ideal görüntüyü yapay zekaya veririz. Yapay zeka bize resimdeki tüm yüzlerin koordinatlarını döndürür.
Ardından bir döngü başlatıp, bulduğu her yüzün üzerine mavi bir dikdörtgen çizer ve yanına "insan" yazdırırız.

In [ ]:
# 2.3 - Yüzleri Tespit Etme ve Orijinal Resimde İşaretleme

# Yapay zekanın yüz koordinatlarını (x,y,w,h) bulması
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.30, minNeighbors=5, minSize=(30, 30))

# Bulunan her yüz için dikdörtgen ve yazı eklenmesi
for (x, y, w, h) in faces:
    cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
    cv2.putText(img, "insan", (x, y+h+20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

### 2.4. Resmi Arayüze (GUI) Uygun Hale Getirme
OpenCV resimleri **BGR** formatında işler, ancak Tkinter arayüzü **RGB** bekler. İnsanların yüzü Avatar filmindeki gibi mavi gözükmesin diye renk sırasını tersine çeviririz.
Son olarak arayüze sığması için resmi 600x400 boyutuna küçültür, `PhotoImage` formatına dönüştürür ve ekrandaki "canvas" (tuval) üzerine bastırırız.

In [ ]:
# 2.4 - Resmi Arayüz (GUI) Formatına Getirme ve Ekrana Basma

# BGR'den RGB'ye dönüşüm
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Pillow ile Tkinter uyumlu hale getirme ve küçültme
img = Image.fromarray(img)
img = img.resize((600, 400), Image.LANCZOS)
img = ImageTk.PhotoImage(img)

# Görüntüyü Tkinter tuvaline yerleştirme (Bunun çalışması için 3. Adımdaki tuvalin oluşturulması gerekir)
# canvas.img = img
# canvas.create_image(0, 0, anchor=tk.NW, image=img)

### 🧩 Şimdi Tüm Parçaları Birleştirelim!
Yukarıda mantığını hücre hücre öğrendiğimiz bu kod bloklarını tek bir çatı (şart/if) ve fonksiyon altında toplayalım.
*(Lütfen aşağıdaki hücreyi çalıştırarak bütün fonksiyonu python belleğine kaydedin:)*

In [ ]:
def open_file():
    # 2.1 - Dosya Seçimi ve İçe Aktarma
    file_path = filedialog.askopenfilename()
    
    if file_path:
        img_array = np.fromfile(file_path, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        
        # 2.2 - Yapay Zekaya Hazırlık (Griye Çevirme ve Işık Dengeleme)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.equalizeHist(gray)
        
        # 2.3 - Yüzleri Tespit Etme ve Orijinal Resimde İşaretleme
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.30, minNeighbors=5, minSize=(30, 30))

        for (x, y, w, h) in faces:
            cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
            cv2.putText(img, "insan", (x, y+h+20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

        # 2.4 - Resmi Arayüz (GUI) Formatına Getirme ve Ekrana Basma
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        img = img.resize((600, 400), Image.LANCZOS)
        img = ImageTk.PhotoImage(img)
        
        canvas.img = img
        canvas.create_image(0, 0, anchor=tk.NW, image=img)

## 3. Adım: Kullanıcı Arayüzünün (GUI) Kurulması

Bu adımda penceremizi çiziyoruz ve içine tuval (`Canvas`) ile butonu (`Button`) yerleştiriyoruz.

**NOT:** Bu hücreyi çalıştırdığınızda bilgisayarınızda bir pencere açılacak ve Notebook bu hücrede asılı (çalışır durumda) kalacaktır. Devam etmek için işiniz bitince o pencereyi kapatın.

In [ ]:
# Ana pencereyi oluşturuyoruz
root = tk.Tk()
root.title("Yüz Tanıma Uygulaması")

# Görüntünün gösterileceği 600x400 piksel boyutunda bir tuval (canvas) oluşturuyoruz
canvas = tk.Canvas(root, width=600, height=400)
canvas.pack()

# Dosya seçme işlemini tetikleyecek butonu oluşturup pencereye ekliyoruz
open_button = tk.Button(root, text="Dosya Seç", command=open_file)
open_button.pack()

# Arayüzün sürekli açık kalmasını ve kullanıcı etkileşimlerini dinlemesini sağlayan sonsuz döngüyü başlatıyoruz
root.mainloop()

---

# 🔄 Programın Çalışma Mantığı ve Kodların Gerçek İşleyiş Sırası

Python kodu sırayla okusa da işleyiş her zaman yukarıdan aşağıya doğru değildir. Aşağıda programı tek parça çalıştırdığınızda arka planda işlemlerin hangi satırlardan geçerek nasıl zıpladığını ve mantığını görebilirsiniz:

### Aşama 1: Arayüzün Çizilmesi ve Bekleme
Python en başta yukarıdaki `open_file` fonksiyonunu sadece hafızasına alır ama içine girip çalıştırmaz (çünkü henüz kimse onu çağırmamıştır). Doğrudan aşağıdaki arayüz kodlarını okumaya başlar. Tuvali ve butonu ekler, sonra `mainloop()` sayesinde program **bekleme/dinleme moduna** geçer.

In [ ]:
root = tk.Tk() # 1- Ana pencere yaratıldı
canvas = tk.Canvas(...) # 2- Tuval eklendi
open_button = tk.Button(..., command=open_file) # 3- Buton eklendi

root.mainloop() # 4- PROGRAM DURDU. Kullanıcının butona tıklaması bekleniyor.

### Aşama 2: Kullanıcı Tıklar ve Fonksiyona Zıplanır
Kullanıcı 'Dosya Seç' butonuna bastığında, butona arayüz aşamasında atadığımız `command=open_file` yetkisi devreye girer. Program döngüden çıkar ve hemen o fonksiyona zıplar. İlk iş olarak dosya seçme penceresi açılır.

In [ ]:
def open_file():
    # 5- Zıplama gerçekleşti! Python artık burada.
    # 6- Kullanıcıya dosya seçme penceresini gösterir ve yolu alır.
    file_path = filedialog.askopenfilename()


### Aşama 3: Görüntünün İşlenmesi ve Yüzlerin Aranması
Dosya seçildiyse, Python o dosyayı okur ve yüz algılama modelinin anlayabileceği hale getirir (Gri tonlama ve Histogram eşitleme).

In [ ]:
    if file_path:
        # 7- Resmi Türkçe karakter engeline takılmadan okur
        img_array = np.fromfile(file_path, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        
        # 8- Renkli resmi siyah-beyaz (gri) tona çevirir
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # 9- Gölgeleri ve ışık patlamalarını dengeleyerek yüzleri belirginleştirir
        gray = cv2.equalizeHist(gray)
        
        # 10- YAPAY ZEKA DEVREDE! Resmin üzerindeki yüzlerin koordinatlarını bulur
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.30, minNeighbors=5, minSize=(30, 30))


### Aşama 4: Bulunan Yüzlerin İşaretlenmesi
Modelin bulduğu her yüz için bir döngü başlar. Orijinal resmin üzerine dikdörtgenler ve 'insan' yazısı eklenir.

In [ ]:
        # 11- Bulunan her yüz için koordinatlar x, y, genişlik(w), yükseklik(h) olarak alınır
        for (x, y, w, h) in faces:
            # 12- Orijinal resmin üzerine mavi bir dikdörtgen çizer
            cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
            # 13- Dikdörtgenin yanına 'insan' yazısı yazar
            cv2.putText(img, "insan", (x, y+h+20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)


### Aşama 5: Resmin Arayüze (GUI) Aktarılması ve Bekleme Modu
Çizim işlemi biten resim, Tkinter'ın gösterebileceği formata dönüştürülür ve ekrana basılır.

In [ ]:
        # 14- OpenCV'nin BGR formatını, arayüzün anladığı RGB formatına çevirir
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        img = img.resize((600, 400), Image.LANCZOS) # 15- Resmi 600x400 boyutuna küçültür
        img = ImageTk.PhotoImage(img)
        
        # 16- Çizilmiş resmi ekrandaki 'canvas' üzerine yerleştirir
        canvas.img = img
        canvas.create_image(0, 0, anchor=tk.NW, image=img)

        # 17- FONKSİYON BİTTİ.


Fonksiyon işini tamamen bitirdiğinde, Python programı başı boş bırakmaz. En baştaki Aşama 1'de beklemeye aldığı o kilit satıra, yani `root.mainloop()` satırına sessizce geri döner ve kullanıcının yeni bir resim seçmesi ihtimaline karşı arayüzü tekrar **dinleme moduna** geçirir.